# 02. Silao Geo-Fenced OSINT Incident Pipeline

Demonstrates bilingual ES/EN news & incident parsing, keyword classification (`traffic_accident`, `road_closure`, `public_safety`, `weather_disruption`), and offline Silao gazetteer geocoding.
Live Google News RSS fetching is opt-in (`FETCH_LIVE=False` by default).

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("BGA_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/bga-matplotlib")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.config import load_config
from src.osint_collector import (
    build_google_news_rss_url,
    build_incident_dataset,
    fetch_rss_feed,
)

DATA_MODE = os.environ.get("BGA_DATA_MODE", "sample").strip().lower()
FETCH_LIVE = os.environ.get("BGA_FETCH_LIVE", "0") == "1"
cfg = load_config(repo_root=REPO_ROOT)
print("Opt-in Google News RSS endpoint:", build_google_news_rss_url())

In [ ]:
fixture_xml = (REPO_ROOT / "tests" / "fixtures" / "sample_feed.xml").read_text(encoding="utf-8")
sample_csv = cfg.sample_dir / "incidents.csv"

incidents_df = build_incident_dataset(
    xml_texts=[fixture_xml],
    file_paths=[sample_csv],
    feed_urls=[build_google_news_rss_url()] if FETCH_LIVE else (),
    use_nominatim=False,
    relevant_only=True,
)

print(f"Collected {len(incidents_df)} deduplicated incidents ({DATA_MODE} pipeline)")
print("Category counts:", incidents_df["category"].value_counts().to_dict())
print("Geocode methods:", incidents_df["geocode_method"].value_counts().to_dict())
print(incidents_df[["date", "hour_int", "category", "severity", "location_name", "geocode_method"]].head(10).to_string(index=False))

In [ ]:
cat_stats = incidents_df.groupby("category", as_index=False).agg(
    count=("incident_id", "count"),
    mean_severity=("severity", "mean"),
)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(cat_stats["category"], cat_stats["count"], color="#2b8cbe")
ax.set_title("OSINT Incidents by Bilingual Category")
ax.set_ylabel("Incident Count")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.close(fig)
print(cat_stats.round(2).to_string(index=False))